# ONC PTWL Colab Pipeline
按顺序运行所有单元。


In [ ]:
# =========================
# User Configuration
# This notebook now uses a hybrid workflow:
# 1. detect rocks
# 2. build a full-grid analysis from the vegetation map
# 3. cut rock polygons directly with canopy polygons
# 4. score the full grid, but only keep outputs inside the canopy-cut rock extent
# The final score rasters and shapefiles are restricted to the rock extent after canopy cutting.
# =========================

DRIVE_ROOT = '/content/drive/MyDrive/ONCMAPPING/DEPLOY'
ORTHO = f'{DRIVE_ROOT}/ORIGINAL_IMG'
CANOPY = f'{DRIVE_ROOT}/CANOPY_IMAGE'
VEG = f'{DRIVE_ROOT}/VEGE_MAP'
OUT_DIR = f'{DRIVE_ROOT}/OUT'

# Model selection
# local_trained1 -> /Model_Weights/best.pt
# local_trained2 -> /Model_Weights/best_DY.pt
# roboflow1      -> /Model_Weights/onc_drone2/9/weights.onnx
# For ONNX models, the script now auto-detects the required square input size.
MODEL_CHOICE = 'roboflow1'   # 'local_trained1', 'local_trained2', 'roboflow1'
MODEL_PATHS = {
    'local_trained1': f'{DRIVE_ROOT}/Model_Weights/best.pt',
    'local_trained2': f'{DRIVE_ROOT}/Model_Weights/best_DY.pt',
    'roboflow1': f'{DRIVE_ROOT}/Model_Weights/onc_drone2/9/weights.onnx',
}
MODEL_PATH = MODEL_PATHS[MODEL_CHOICE]
MODEL_BACKEND = 'auto'      # 'auto', 'pt', 'onnx'

# Run mode
# full           -> detect rocks, then score full-grid cells clipped to rock extent
# detection_only -> only write rocks.shp
# habitat_only   -> skip detection and use EXISTING_ROCKS directly
RUN_MODE = 'full'
EXISTING_ROCKS = ''

# Detection settings
TILE_SIZE = 512
OVERLAP = 128
CONF = 0.25
IOU_NMS = 0.35
MAX_TILES = None
MODEL_IMGSZ = 640  # used for .pt models; .onnx models are auto-adjusted if needed
TARGET_CLASS_NAMES = ['rock']
GREEN_FILTER = False
GREEN_THRESHOLD = 0.35
GREEN_MARGIN = 12.0

# Optional size-bin outputs
SIZE_BINS_ENABLED = False
SIZE_BINS = '10,40,100'
SIZE_METRIC = 'max_side_cm'
MANUAL_CM_PER_PIXEL = None
WRITE_SIZE_BIN_SHAPEFILES = True
HABITAT_SIZE_BIN = ''

# Rock-overlay scoring settings
BLOCK_SIZE = '1'
SCORE_SCALING = 'absolute'
VEGETATION_WEIGHT = 0.7
ROCK_WEIGHT = 0.3
ROCK_PERCENTILE = 95.0
ROCK_CAP = None
ROCK_ASSIGNMENT = 'centroid'

SMOKE_TEST = False
RUN_NAME = ''
SPECIAL_POINTS = [
    ('P1', -35.398418, 149.048551),
    ('P2', -35.399366, 149.048177),
    ('P3', -35.400982, 149.048679),
]
SPECIAL_RADIUS_M = 11.3


In [ ]:
import subprocess
import sys

core_deps = [
    'geopandas',
    'rasterio',
    'shapely',
    'fiona',
    'pyproj',
    'rtree',
    'ultralytics',
    'pandas',
    'numpy',
    'matplotlib',
    'affine',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + core_deps)
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'])
    print('Installed onnxruntime-gpu')
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'])
    print('Installed onnxruntime')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import torch

print('torch =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu_name =', torch.cuda.get_device_name(0))
print('model_choice =', MODEL_CHOICE)
print('model_path =', MODEL_PATH)
print('model_backend =', MODEL_BACKEND)
print('model_ext =', os.path.splitext(MODEL_PATH)[1].lower())


In [ ]:
from pathlib import Path
import sys

COLAB_SCRIPT = Path(f'{DRIVE_ROOT}/stone_pipeline_colab.py')
print('COLAB_SCRIPT =', COLAB_SCRIPT)

if not COLAB_SCRIPT.exists():
    raise FileNotFoundError(f'找不到脚本文件: {COLAB_SCRIPT}')

script_dir = str(COLAB_SCRIPT.parent)
print('script_dir =', script_dir)

if script_dir not in sys.path:
    sys.path.insert(0, script_dir)

print('sys.path[0] =', sys.path[0])

from stone_pipeline_colab import log, resolve_input_path, resolve_input_paths, run_pipeline
print('stone_pipeline_colab 导入成功')

In [ ]:
# Run the hybrid full-grid + rock-clipped workflow for one file or for every ortho image under ORTHO
ortho_paths = resolve_input_paths(ORTHO, ['.tif', '.tiff', '.img', '.jp2', '.vrt'], 'ORTHO')
canopy_path = resolve_input_path(CANOPY, ['.shp'], 'CANOPY')
veg_path = resolve_input_path(VEG, ['.tif', '.tiff', '.img', '.jp2', '.vrt'], 'VEG')
weights_path = resolve_input_path(MODEL_PATH, ['.pt', '.onnx'], 'MODEL_PATH')
existing_rocks_path = resolve_input_path(EXISTING_ROCKS, ['.shp'], 'EXISTING_ROCKS') if EXISTING_ROCKS else ''

effective_max_tiles = 1 if SMOKE_TEST and MAX_TILES is None else MAX_TILES
all_outputs = []

for idx, ortho_path in enumerate(ortho_paths, start=1):
    image_name = os.path.splitext(os.path.basename(ortho_path))[0]
    log(f'[{idx}/{len(ortho_paths)}] Processing image: {image_name}')
    result = run_pipeline(
        ortho=ortho_path,
        canopy=canopy_path,
        veg=veg_path,
        weights=weights_path,
        out_dir=OUT_DIR,
        run_mode=RUN_MODE,
        existing_rocks=existing_rocks_path,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        conf=CONF,
        iou_nms=IOU_NMS,
        max_tiles=effective_max_tiles,
        model_imgsz=MODEL_IMGSZ,
        model_backend=MODEL_BACKEND,
        target_class_names=TARGET_CLASS_NAMES,
        green_filter=GREEN_FILTER,
        green_threshold=GREEN_THRESHOLD,
        green_margin=GREEN_MARGIN,
        size_bins_enabled=SIZE_BINS_ENABLED,
        size_bins=SIZE_BINS,
        size_metric=SIZE_METRIC,
        manual_cm_per_pixel=MANUAL_CM_PER_PIXEL,
        write_size_bin_shapefiles=WRITE_SIZE_BIN_SHAPEFILES,
        habitat_size_bin=HABITAT_SIZE_BIN,
        block_size=BLOCK_SIZE,
        score_scaling=SCORE_SCALING,
        vegetation_weight=VEGETATION_WEIGHT,
        rock_weight=ROCK_WEIGHT,
        rock_percentile=ROCK_PERCENTILE,
        rock_cap=ROCK_CAP,
        rock_assignment=ROCK_ASSIGNMENT,
        image_name=image_name,
        run_name=RUN_NAME,
        special_points=SPECIAL_POINTS,
        special_radius_m=SPECIAL_RADIUS_M,
    )
    all_outputs.extend(result['outputs'])

print('Completed')
for item in all_outputs:
    print(item)
